In [1]:
#!pip install uszipcode
#!pip install 'sqlalchemy_mate < 2.0.0.1'
#!pip install us

In [2]:
import pandas as pd
from uszipcode import SearchEngine
import us

/opt/anaconda3/lib/python3.13/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
cdc_24 = pd.read_csv('../data/cleaned/cdc_cleaned_2024.csv')
cdc_24

,county,fips,Sex,Sex Code,year,Year Code,Cause of death,Cause of death Code,Place of Death,Place of Death Code,deaths
0,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0
1,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0
2,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to other ...,X44,Decedent's home,4.0,23.0
3,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to other ...,X44,Other,7.0,12.0
4,"Jefferson County, AL",1073.0,Male,M,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Medical Facility - Outpatient or ER,2.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...
1475,"Milwaukee County, WI",55079.0,Male,M,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,44.0
1476,"Milwaukee County, WI",55079.0,Male,M,2024.0,2024.0,Accidental poisoning by and exposure to other ...,X44,Decedent's home,4.0,73.0
1477,"Milwaukee County, WI",55079.0,Male,M,2024.0,2024.0,Accidental poisoning by and exposure to other ...,X44,Other,7.0,43.0
1478,"Racine County, WI",55101.0,Male,M,2024.0,2024.0,Accidental poisoning by and exposure to other ...,X44,Decedent's home,4.0,11.0


In [4]:
facilities = pd.read_csv('../data/cleaned/samhsa_facilities_zip.csv')
facilities

,state,zip,facility_count
0,AK,99501,2
1,AK,99503,6
2,AK,99507,1
3,AK,99508,10
4,AK,99518,2
...,...,...,...
5344,WY,82941,1
5345,WY,83001,1
5346,WY,83101,1
5347,WY,83110,1


In [5]:
census_24 = pd.read_csv('../data/cleaned/census_acs_2024.csv')
census_24

,county_name,median_income,population,state,county,poverty_rate,unemployment_rate,fips
0,"Autauga County, Alabama",72481,59947,1,1,11.3,2.4,1001
1,"Baldwin County, Alabama",78775,246989,1,3,10.1,3.0,1003
2,"Barbour County, Alabama",46042,24643,1,5,21.4,7.8,1005
3,"Bibb County, Alabama",52541,22130,1,7,22.5,12.1,1007
4,"Blount County, Alabama",64190,59518,1,9,12.9,5.0,1009
...,...,...,...,...,...,...,...,...
3217,"Vega Baja Municipio, Puerto Rico",24244,53892,72,145,41.1,11.0,72145
3218,"Vieques Municipio, Puerto Rico",19803,8078,72,147,59.1,6.1,72147
3219,"Villalba Municipio, Puerto Rico",26286,21556,72,149,38.9,11.4,72149
3220,"Yabucoa Municipio, Puerto Rico",22944,29400,72,151,47.7,8.7,72151


## To begin this notebook I need to make sure that my county columns are able to be merged on for all of my datasets.

In [6]:
search = SearchEngine()

facilities["county"] = (
    facilities["zip"]
    .apply(
        lambda z:
        search.by_zipcode(z).county
        if search.by_zipcode(z)
        else None
    )
)

facilities.head()

,state,zip,facility_count,county
0,AK,99501,2,Anchorage Municipality
1,AK,99503,6,Anchorage Municipality
2,AK,99507,1,Anchorage Municipality
3,AK,99508,10,Anchorage Municipality
4,AK,99518,2,Anchorage Municipality


In [7]:
facilities.sample(5)

,state,zip,facility_count,county
3122,NC,28390,1,Harnett County
4703,TX,79905,2,El Paso County
1053,FL,32904,2,Brevard County
2915,MS,39083,1,Copiah County
5179,WI,54017,1,St. Croix County


In [8]:
facilities['county_state'] = facilities['county'] + ', ' + facilities['state']
facilities.head()

,state,zip,facility_count,county,county_state
0,AK,99501,2,Anchorage Municipality,"Anchorage Municipality, AK"
1,AK,99503,6,Anchorage Municipality,"Anchorage Municipality, AK"
2,AK,99507,1,Anchorage Municipality,"Anchorage Municipality, AK"
3,AK,99508,10,Anchorage Municipality,"Anchorage Municipality, AK"
4,AK,99518,2,Anchorage Municipality,"Anchorage Municipality, AK"


In [9]:
facilities = facilities.groupby('county_state')['facility_count'].sum().reset_index()
facilities

,county_state,facility_count
0,", GU",1
1,", MP",1
2,"Acadia Parish, LA",1
3,"Accomack County, VA",2
4,"Ada County, ID",36
...,...,...
2082,"Yuba County, CA",4
2083,"Yukon-Koyukuk Census Area, AK",1
2084,"Yuma County, AZ",15
2085,"Yuma County, CO",1


In [10]:
cdc_24["fips"] = (
    cdc_24["fips"]
    .astype(str)
    .str.zfill(5)
)

census_24["fips"] = (
    census_24["fips"]
    .astype(str)
    .str.zfill(5)
)

In [11]:
#rename state and county column to mergre on
cdc_24 = cdc_24.rename(columns= {'county': 'county_state'})
cdc_24.head(2)

,county_state,fips,Sex,Sex Code,year,Year Code,Cause of death,Cause of death Code,Place of Death,Place of Death Code,deaths
0,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0
1,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0


In [12]:
census_24 = census_24.rename(columns= {'county_name': 'county_state'})
census_24.head(2)

,county_state,median_income,population,state,county,poverty_rate,unemployment_rate,fips
0,"Autauga County, Alabama",72481,59947,1,1,11.3,2.4,01001
1,"Baldwin County, Alabama",78775,246989,1,3,10.1,3.0,01003


In [13]:
def safe_state_lookup(state_name):
    """Safely lookup state abbreviation, return original name if not found"""
    try:
        state = us.states.lookup(state_name.strip())  # Remove extra whitespace
        return state.abbr if state else state_name  # Return abbr if found, else original name
    except:
        return state_name  # Return original name if any error occurs

census_24['county_state'] = (
    census_24['county_state']
    .str.split(', ')
    .apply(lambda x: f"{x[0]}, {safe_state_lookup(x[1])}")
)

census_24.head(2)

,county_state,median_income,population,state,county,poverty_rate,unemployment_rate,fips
0,"Autauga County, AL",72481,59947,1,1,11.3,2.4,01001
1,"Baldwin County, AL",78775,246989,1,3,10.1,3.0,01003


In [14]:
census_24['county_state'].value_counts()

county_state
Autauga County, AL     1
Custer County, OK      1
Carter County, OK      1
Cherokee County, OK    1
Choctaw County, OK     1
                      ..
Meade County, KY       1
Menifee County, KY     1
Mercer County, KY      1
Metcalfe County, KY    1
Yauco Municipio, PR    1
Name: count, Length: 3222, dtype: int64

In [15]:
facilities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   county_state    2087 non-null   object
 1   facility_count  2087 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 32.7+ KB


## Merging 

In [16]:
facilities.head(2)

,county_state,facility_count
0,", GU",1
1,", MP",1


In [17]:
master_raw = cdc_24.merge(census_24, on='county_state') \
               .merge(facilities, on='county_state', how= 'left')
master_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1437 entries, 0 to 1436
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   county_state         1437 non-null   object 
 1   fips_x               1437 non-null   object 
 2   Sex                  1437 non-null   object 
 3   Sex Code             1437 non-null   object 
 4   year                 1437 non-null   float64
 5   Year Code            1437 non-null   float64
 6   Cause of death       1437 non-null   object 
 7   Cause of death Code  1437 non-null   object 
 8   Place of Death       1437 non-null   object 
 9   Place of Death Code  1437 non-null   float64
 10  deaths               1437 non-null   float64
 11  median_income        1437 non-null   int64  
 12  population           1437 non-null   int64  
 13  state                1437 non-null   int64  
 14  county               1437 non-null   int64  
 15  poverty_rate         1437 non-null   f

In [18]:
master_raw.shape

(1437, 19)

In [19]:
master_raw.head(2)

,county_state,fips_x,Sex,Sex Code,year,Year Code,Cause of death,Cause of death Code,Place of Death,Place of Death Code,deaths,median_income,population,state,county,poverty_rate,unemployment_rate,fips_y,facility_count
0,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0,66388,667755,1,73,15.7,4.7,01073,7.0
1,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0,66388,667755,1,73,15.7,4.7,01073,7.0


### Now that I have my raw master data frame I can clean it up

In [20]:
master_raw = master_raw.drop(columns=['Year Code', 'state', 'county', 'fips_y'])

In [21]:
master_raw.head(2)

,county_state,fips_x,Sex,Sex Code,year,Cause of death,Cause of death Code,Place of Death,Place of Death Code,deaths,median_income,population,poverty_rate,unemployment_rate,facility_count
0,"Jefferson County, AL",1073.0,Female,F,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0,66388,667755,15.7,4.7,7.0
1,"Jefferson County, AL",1073.0,Female,F,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0,66388,667755,15.7,4.7,7.0


In [22]:
master = master_raw.rename(columns={
    'fips_x': 'fips',
    'Sex': 'sex',
    'Sex Code': 'sex_code',
    'Cause of death': 'cause_of_death',
    'Cause of death Code': 'cod_code',
    'Place of Death': 'place_of_death',
    'Place of Death Code': 'place_of_death_code'
})


In [25]:
#last clean up 
#last clean up 
master['fips'] = (
    master['fips']
    .fillna(0)
    .astype(float)  # First convert to float to handle any string decimals
    .astype(int)    # Then convert to int (this removes decimal places)
    .astype(str)    # Convert to string for zfill formatting
    .str.zfill(5)   # Pad with leading zeros to make 5 digits
)

In [26]:
master.head(2)

,county_state,fips,sex,sex_code,year,cause_of_death,cod_code,place_of_death,place_of_death_code,deaths,median_income,population,poverty_rate,unemployment_rate,facility_count
0,"Jefferson County, AL",01073,Female,F,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0,66388,667755,15.7,4.7,7.0
1,"Jefferson County, AL",01073,Female,F,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0,66388,667755,15.7,4.7,7.0


In [27]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1437 entries, 0 to 1436
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   county_state         1437 non-null   object 
 1   fips                 1437 non-null   object 
 2   sex                  1437 non-null   object 
 3   sex_code             1437 non-null   object 
 4   year                 1437 non-null   float64
 5   cause_of_death       1437 non-null   object 
 6   cod_code             1437 non-null   object 
 7   place_of_death       1437 non-null   object 
 8   place_of_death_code  1437 non-null   float64
 9   deaths               1437 non-null   float64
 10  median_income        1437 non-null   int64  
 11  population           1437 non-null   int64  
 12  poverty_rate         1437 non-null   float64
 13  unemployment_rate    1437 non-null   float64
 14  facility_count       1308 non-null   float64
dtypes: float64(6), int64(2), object(7)
mem

In [28]:
master.shape

(1437, 15)

## Now that I have my Master Data Frame, will add come variables that I plan to work with.

In [29]:
#analysis variable
master["overdose_rate"] = (
    master["deaths"]
    /
    master["population"]
) * 100000

In [30]:
master["facility_rate"] = (
    master["facility_count"]
    /
    master["population"]
) * 100000

In [31]:
master.head()

,county_state,fips,sex,sex_code,year,cause_of_death,cod_code,place_of_death,place_of_death_code,deaths,median_income,population,poverty_rate,unemployment_rate,facility_count,overdose_rate,facility_rate
0,"Jefferson County, AL",01073,Female,F,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0,66388,667755,15.7,4.7,7.0,2.246333,1.048289
1,"Jefferson County, AL",01073,Female,F,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0,66388,667755,15.7,4.7,7.0,1.797066,1.048289
2,"Jefferson County, AL",01073,Female,F,2024.0,Accidental poisoning by and exposure to other ...,X44,Decedent's home,4.0,23.0,66388,667755,15.7,4.7,7.0,3.444377,1.048289
3,"Jefferson County, AL",01073,Female,F,2024.0,Accidental poisoning by and exposure to other ...,X44,Other,7.0,12.0,66388,667755,15.7,4.7,7.0,1.797066,1.048289
4,"Jefferson County, AL",01073,Male,M,2024.0,Accidental poisoning by and exposure to narcot...,X42,Medical Facility - Outpatient or ER,2.0,10.0,66388,667755,15.7,4.7,7.0,1.497555,1.048289


Now lets save it as a csv

In [32]:
master.to_csv('../data/master_df.csv', index=False)